In [ ]:
import pickle
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt

with open("demo_2x2_ctrl_limited/demo_2x2_ctrl_limited.pkl", "rb") as f:
    results = pickle.load(f)

t_arr = np.array([r["t"] for r in results])
n_turbines = 4

In [ ]:
# Farm layout (from 2x2.fstf): wind flows in +X direction
#   row 0 (top,    Y=630): T2 (upstream)  | T3 (downstream)
#   row 1 (bottom, Y=0  ): T0 (upstream)  | T1 (downstream)
turb_to_ax = {0: (0, 0), 1: (0, 1), 2: (1, 0), 3: (1, 1)}
fig_yaw, axes_yaw = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
fig_yaw.suptitle("Yaw Dashboard — wind dir / nacelle yaw / platform rdz / error (per turbine)", fontsize=13)

for i in range(n_turbines):
    row, col = turb_to_ax[i]
    ax = axes_yaw[row][col]

    wind_dir = np.array([r[f"wind_dir_T{i}"]      for r in results]) - 270         # convert to 0°=from left, +CCW
    yaw_meas = np.array([r[f"yaw_T{i}"]           for r in results])
    rdz      = np.array([r[f"platform_rdz_T{i}"]  for r in results])
    error    = wind_dir - (yaw_meas + rdz)  
    error2   = wind_dir

    ax.plot(t_arr, wind_dir, label="Wind dir (deg)",    color="steelblue")
    ax.plot(t_arr, yaw_meas, label="Nacelle yaw (deg)", color="darkorange",  linestyle="--")
    ax.plot(t_arr, rdz,      label="Platform rdz (deg)",color="tab:green",   linestyle=":")
    ax.plot(t_arr, error,    label="Error (deg)",        color="tab:red",     linestyle="-.")
    ax.plot(t_arr, error2,   label="Error2 (deg)",       color="tab:purple",  linestyle="-.")
    ax.set_title(f"WT{i}")
    ax.set_ylabel("Angle (deg)")
    if row == 1:
        ax.set_xlabel("Time (s)")
    ax.legend(fontsize=7)
    ax.grid(True)
fig_yaw.tight_layout()
plt.show()

In [ ]:
# Farm layout (from 2x2.fstf): wind flows in +X direction
#   row 0 (top,    Y=630): T2 (upstream)  | T3 (downstream)
#   row 1 (bottom, Y=0  ): T0 (upstream)  | T1 (downstream)

# plot the figure that compare wind direction and nacelle yaw for each turbine
turb_to_ax_cmp = {0: (0, 0), 1: (0, 1), 2: (1, 0), 3: (1, 1)}
fig_cmp, axes_cmp = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
# fig_cmp.suptitle("Wind Direction vs Nacelle Yaw (per turbine)", fontsize=13)

for i in range(n_turbines):
    row, col = turb_to_ax_cmp[i]
    ax = axes_cmp[row][col]

    wind_dir = np.array([r[f"wind_dir_T{i}"] for r in results]) - 270  # target heading, "yaw" frame
    yaw_meas = np.array([r[f"yaw_T{i}"]      for r in results])        # measured nacelle yaw (deg)

    ax.plot(t_arr, wind_dir, label="Wind dir (deg)",    color="steelblue")
    ax.plot(t_arr, yaw_meas, label="Nacelle yaw (deg)", color="darkorange", linestyle="--")
    ax.set_title(f"WT{i}", fontsize=14)
    ax.set_ylabel("Angle (deg)", fontsize=14)
    if row == 1:
        ax.set_xlabel("Time (s)", fontsize=14)
    ax.legend(fontsize=14)
    ax.grid(True)
fig_cmp.tight_layout()
plt.show()

In [ ]:
# vtk snapshot
t = np.array([150, 300, 400, 700, 800])
vtk_time = np.round(t / 3)
vtk_file_path = "D:\\2_PhD_UBC\\Code\\PyConnectFastFarm\\demo\\examples\\vtk_ff_limited"

# Turbine positions from 2x2.fstf (WT_X, WT_Y), labeled to match WT0-WT3 (top) / WT2-WT3 (bottom) convention
turbine_pos = {"WT0": (5, 630), "WT1": (635, 630), "WT2": (5, 0), "WT3": (635, 0)}

def load_vtk_speed(vtk_path):
    mesh = pv.read(vtk_path)
    dims = mesh.dimensions
    x  = mesh.points[:, 0].reshape(dims[1], dims[0])
    y  = mesh.points[:, 1].reshape(dims[1], dims[0])
    Vx = mesh['Velocity'][:, 0].reshape(dims[1], dims[0])
    Vy = mesh['Velocity'][:, 1].reshape(dims[1], dims[0])
    speed = np.sqrt(Vx**2 + Vy**2)
    return x, y, speed

# Load each selected snapshot
snapshots = []
for idx in vtk_time.astype(int):
    fpath = f"{vtk_file_path}/2x2.Low.DisXY001.{idx:04d}.vtk"
    snapshots.append(load_vtk_speed(fpath))

# Shared color scale across all snapshots
all_speed = np.concatenate([s[2].ravel() for s in snapshots])
vmin, vmax = np.nanpercentile(all_speed, [2, 98])

# One individual plot per snapshot
from mpl_toolkits.axes_grid1 import make_axes_locatable

for t_i, idx, (x, y, speed) in zip(t, vtk_time.astype(int), snapshots):
    fig, ax = plt.subplots(figsize=(8, 5))
    pcm = ax.pcolormesh(x, y, speed, cmap='coolwarm', shading='auto', vmin=vmin, vmax=vmax)
    for name, (px, py) in turbine_pos.items():
        ax.plot(px, py, 'ko')
        ax.text(px + 10, py + 10, name, fontsize=9)
    ax.set_aspect('equal')
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_title(f"t = {t_i} s")

    # make the colorbar the same height as the (aspect-constrained) image
    fig.canvas.draw()
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=0.1)
    fig.colorbar(pcm, cax=cax, label="Wind speed [m/s]")

    fig.tight_layout()
    plt.show()